In [ ]:
import pandas as pd

# escolher o número de componentes
# número de componentes é hiperparâmetro
df_normal = pd.read_parquet("../df_normal.parquet")
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Dados normalizados
X_db = df_normal.copy()

# Se existir alguma coluna de cluster anterior, remover
X_db = X_db.drop("cluster", axis=1, errors="ignore")
X_db = X_db.drop("cluster_dbscan", axis=1, errors="ignore")

# Amostra porque tens muitos dados
X_db_amostra = X_db.sample(n=80000, random_state=42)

# Testar vários valores de eps
eps_values = [0.2, 0.5, 0.8, 1, 1.5, 2, 3, 4, 5]

for eps in eps_values:
    dbscan = DBSCAN(
        eps=eps,
        min_samples=10,
        n_jobs=-1
    )

    labels_dbscan = dbscan.fit_predict(X_db_amostra)

    n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    n_outliers = list(labels_dbscan).count(-1)

    print("eps:", eps)
    print("Número de clusters:", n_clusters)
    print("Número de outliers:", n_outliers)
    print("Percentagem de outliers:", round(n_outliers / len(labels_dbscan) * 100, 2), "%")
    print("-" * 40)
    
# escolher ep=3

Fit modelo

In [ ]:
from sklearn.cluster import DBSCAN

dbscan_final = DBSCAN(
    eps=3.3,
    min_samples=10,
    n_jobs=-1
)

labels_dbscan = dbscan_final.fit_predict(X_db_amostra)

df_dbscan_f = X_db_amostra.copy()
df_dbscan_f["cluster_dbscan"] = labels_dbscan

print(df_dbscan_f["cluster_dbscan"].value_counts())

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# Variáveis
X_db_metricas = df_dbscan_f.drop("cluster_dbscan", axis=1)
y_db_metricas = df_dbscan_f["cluster_dbscan"]

# Número de clusters, ignorando outliers (-1)
n_clusters = len(set(y_db_metricas)) - (1 if -1 in y_db_metricas.values else 0)

# Número de outliers
n_outliers = (y_db_metricas == -1).sum()

# Percentagem de outliers
perc_outliers = n_outliers / len(y_db_metricas) * 100

print("Métricas gerais do DBSCAN:")
print("Número de clusters:", n_clusters)
print("Número de outliers:", n_outliers)
print("Percentagem de outliers:", round(perc_outliers, 2), "%")

print("\nNúmero de elementos por cluster:")
print(y_db_metricas.value_counts())

In [ ]:
# Remover outliers para calcular métricas
mask = y_db_metricas != -1

X_sem_outliers = X_db_metricas.loc[mask]
y_sem_outliers = y_db_metricas.loc[mask]

# Só dá para calcular métricas se existirem pelo menos 2 clusters
if len(set(y_sem_outliers)) >= 2:

    silhouette = silhouette_score(
        X_sem_outliers,
        y_sem_outliers,
        sample_size=min(3000, len(X_sem_outliers)),
        random_state=42
    )

    davies = davies_bouldin_score(X_sem_outliers, y_sem_outliers)
    calinski = calinski_harabasz_score(X_sem_outliers, y_sem_outliers)

    print("\nMétricas de qualidade do DBSCAN:")
    print("Silhouette Score:", silhouette)
    print("Davies-Bouldin Score:", davies)
    print("Calinski-Harabasz Score:", calinski)

else:
    print("\nO DBSCAN não criou clusters suficientes para calcular Silhouette, Davies-Bouldin e Calinski-Harabasz.")